# 🧠 AI Adaptive Onboarding Engine — Kaggle Training Notebook

**Multi-Task Siamese Transformer** built from scratch in PyTorch.
- 2× T4 GPUs | Mixed Precision (fp16) | DataParallel
- Tasks: Resume–JD similarity + Multi-label skill extraction


In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
!pip install -q PyMuPDF streamlit tqdm

In [ ]:
# ── 2. Upload / extract project ───────────────────────────────────────────
# Option A: If you've zipped the project and added it as a Kaggle dataset:
import zipfile, os
# zipfile.ZipFile('/kaggle/input/ai-onboarding/ai_onboarding_engine.zip').extractall('/kaggle/working/')

# Option B: Clone from GitHub (if public):
# !git clone https://github.com/YOUR_REPO/ai_onboarding_engine /kaggle/working/ai_onboarding_engine

os.chdir('/kaggle/working/ai_onboarding_engine')
print('Working dir:', os.getcwd())

In [ ]:
# ── 3. Verify GPU setup ───────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

In [ ]:
# ── 4. Quick sanity checks ────────────────────────────────────────────────

# Test tokenizer
from utils.tokenizer import WordTokenizer
tok = WordTokenizer(500)
tok.build(['python machine learning docker kubernetes aws', 'data science nlp transformer'], min_freq=1)
print('Tokenizer vocab size:', tok.vocab_size)
ids = tok.encode('python and machine learning', max_len=16)
print('Encoded:', ids)

# Test model forward pass
from models.transformer import SiameseOnboardingModel, ModelConfig
cfg   = ModelConfig(vocab_size=tok.vocab_size)
model = SiameseOnboardingModel(cfg)
print(f'Model parameters: {model.count_parameters():,}')

import torch
r = torch.randint(4, tok.vocab_size, (2, 32))
j = torch.randint(4, tok.vocab_size, (2, 32))
out = model(r, j)
print('Similarity shape:', out['similarity'].shape)
print('Skill logits shape:', out['resume_skills'].shape)

In [ ]:
# ── 5. Train ──────────────────────────────────────────────────────────────
from training.train import TrainConfig, train

cfg = TrainConfig(
    n_samples    = 12_000,
    epochs       = 20,
    batch_size   = 32,      # per GPU; effective = 64 on 2× T4
    hidden_dim   = 256,
    num_layers   = 3,
    num_heads    = 4,
    max_len      = 256,
    lr           = 3e-4,
    fp16         = True,
    lambda_sim   = 1.0,
    lambda_skill = 0.5,
    save_dir     = '/kaggle/working/checkpoints',
    log_interval = 20,
)

train(cfg)

In [ ]:
# ── 6. Inference test ────────────────────────────────────────────────────
from training.inference import OnboardingEngine

engine = OnboardingEngine(
    checkpoint='/kaggle/working/checkpoints/best_model.pt',
    tokenizer='/kaggle/working/checkpoints/tokenizer.json',
)

resume = """
Senior Software Engineer with 5 years of experience.
Skilled in Python, Django, REST API, SQL, PostgreSQL, Git, Linux.
Some experience with Docker and AWS.
"""

jd = """
Looking for a Backend ML Engineer.
Requirements: Python, PyTorch, machine learning, deep learning,
Docker, Kubernetes, AWS, mlflow, CI/CD, PostgreSQL.
"""

result = engine.analyze(resume_text=resume, jd_text=jd)

print(f"Match Score : {result['match_score']:.0%}")
print(f"Resume skills: {result['resume_skills']}")
print(f"JD skills:     {result['jd_skills']}")
print(f"Missing:       {result['gap_analysis']['missing']}")
print(f"Total hours:   {result['roadmap']['total_hours']}")
print("\nPhases:")
for phase, skills in result['roadmap']['phases'].items():
    print(f'  {phase}:', skills)

In [ ]:
# ── 7. Plot training curves ──────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('/kaggle/working/checkpoints/training_log.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(log['epoch'], log['train_loss'], label='Train')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(log['epoch'], log['train_sim_acc'], label='Train')
axes[1].plot(log['epoch'], log['val_sim_acc'],   label='Val')
axes[1].set_title('Similarity Accuracy'); axes[1].legend()

axes[2].plot(log['epoch'], log['train_skill_f1'], label='Train')
axes[2].plot(log['epoch'], log['val_skill_f1'],   label='Val')
axes[2].set_title('Skill F1'); axes[2].legend()

plt.tight_layout(); plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()